In [ ]:
# =====================================================================
# CELLA IMPORTAZIONI LIBRERIE PER MAC
# =====================================================================
import os
import glob
import logging
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from keras.models import load_model

# Silenziamo i log inutili del Mac
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Blocca tutto tranne gli errori fatali
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' 
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# NOTA: Rimosso 'TF_CUDNN_USE_AUTOTUNE' perché sul tuo Mac non serve

import tensorflow as tf

# Altri silenziatori di log 
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

# =====================================================================
# MODIFICA 1: DISABILITARE LA GPU PER EVITARE I GRADIENTI NaN (ESPLOSIONE DELLA LOSS)
# =====================================================================
# Diciamo a TensorFlow di "nascondere" la GPU M1 (gestita da tensorflow-metal).
tf.config.set_visible_devices([], 'GPU')

# Riga di controllo per essere sicuri al 100% che abbia funzionato
print("Dispositivi di calcolo attivi:", tf.config.get_visible_devices())
# =====================================================================

# Import di Keras (lasciati identici a quelli del tuo collega)
from keras import layers, models, losses
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

In [ ]:
# ==============================================================================
# 1. DATA ENGINE V9 (Caricamento Globale in RAM)
# ==============================================================================
def load_and_process_all_files(file_list, alpha=0.20):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        # Inizializza il background per l'EMA
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        # L'EMA viene calcolato qui, una volta per tutte, in perfetto ordine cronologico!
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        # Flatten delle coordinate e concatenazione con la mask (12 valori totali)
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato. ({T} frame pre-calcolati)")

    # Uniamo tutte le liste in due immensi tensori Numpy pronti per la GPU
    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y

# ==============================================================================
# 2. SPLIT STRATIFICATO RIGOROSO
# ==============================================================================
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]

#tutti_i_file = glob.glob("dataset/data/*.npz")
tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

# ==============================================================================
# 3. ESECUZIONE DEL MOTORE (ATTENZIONE: Ci metterà ~1 minuto a caricare tutto in RAM)
# ==============================================================================
print("\n--- PREPARAZIONE TRAINING SET ---")
X_train, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val, Y_val = load_and_process_all_files(val_files)

print("\n==================================================")
print(f"DATI TOTALI PRONTI IN RAM!")
print(f"Totale FRAME individuali di Train:      {X_train.shape[0]}")
print(f"Totale FRAME individuali di Validation: {X_val.shape[0]}")
print("==================================================")

In [ ]:
def masked_mse(y_true, y_pred):
    """
    Calcola l'errore sulle coordinate (MSE). 
    In futuro potremo azzerarlo se la maschera è 0.
    """
    return losses.mean_squared_error(y_true, y_pred)

In [ ]:
def true_masked_mse(y_true_combined, y_pred_coords):
    y_true_coords = y_true_combined[:, :8]
    mask_1d = y_true_combined[:, 8:] 
    mask_2d = tf.repeat(mask_1d, 2, axis=1) 
    
    raw_mse = tf.square(y_true_coords - y_pred_coords)
    
    masked_mse = raw_mse * mask_2d
    
    # FIX: axis=1 calcola l'errore per singolo frame nel batch
    sum_mse_per_frame = tf.reduce_sum(masked_mse, axis=1)
    valid_elements_per_frame = tf.reduce_sum(mask_2d, axis=1) + 1e-6
    
    return sum_mse_per_frame / valid_elements_per_frame

def true_masked_rmse_metres(y_true_combined, y_pred_coords):
    y_true_coords = y_true_combined[:, :8]
    mask_1d = y_true_combined[:, 8:] 
    mask_2d = tf.repeat(mask_1d, 2, axis=1) 
    
    raw_mse = tf.square(y_true_coords - y_pred_coords)
    masked_mse = raw_mse * mask_2d
    
    # FIX: axis=1 calcola lo scarto quadratico per singolo frame
    sum_mse_per_frame = tf.reduce_sum(masked_mse, axis=1)
    valid_elements_per_frame = tf.reduce_sum(mask_2d, axis=1) + 1e-6
    
    # Ritorna la radice quadrata (RMSE) che corrisponde ai metri reali
    return tf.sqrt(sum_mse_per_frame / valid_elements_per_frame)

In [ ]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V1-Heavy (Struttura Romana, Potenza Toscana) nuova
# ==============================================================================
def build_eeai_model_v1_heavy(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # --- BLOCCO 1: Estrazione Base ---
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_1a")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x)
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_1b")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) # Dimensione: 120 -> 60
    
    # --- BLOCCO 2: Livello Intermedio ---
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_1")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_3")(x)
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_2")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_4")(x) # Dimensione: 60 -> 30
    
    # --- BLOCCO 3: Feature di Alto Livello ---
    x = layers.Conv2D(128, (1, 3), padding='same', activation='relu', name="conv_3a")(x)
    
    # Compattazione salvavita per ESP32
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    
    # --- COLLO DI BOTTIGLIA PIÙ PROFONDO ---
    x = layers.Dense(128, activation='relu', name="features_deep")(x)
    x = layers.Dropout(0.3, name="drop_features")(x) 
    common_feat = layers.Dense(64, activation='relu', name="features")(x)

    # --- OUTPUT MULTI-HEAD INVARIATI ---
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    return models.Model(inputs=inputs, outputs=[coords_output, mask_output], name="EEAI_Net_V1_Heavy")

# Inizializzazione del nuovo modello
model_heavy = build_eeai_model_v1_heavy()

# Compilazione nuova
model_heavy.compile(
    optimizer='adam',
    loss={"coords_head": true_masked_mse, "mask_head": "binary_crossentropy"}, 
    loss_weights={"coords_head": 1.0, "mask_head": 0.5},
    metrics={
        #"coords_head": [tf.keras.metrics.RootMeanSquaredError(name="metres")],
        #"mask_head": ["accuracy"]
        "coords_head": [true_masked_rmse_metres],
        "mask_head": [tf.keras.metrics.BinaryAccuracy(name="bin_acc")]
    }
)

# --- 1. IL CHECKPOINT ORIGINALE ---
checkpoint_heavy = ModelCheckpoint(
    "eeai_best_model_romano_heavy.keras", 
    monitor="val_loss", 
    save_best_only=True, 
    verbose=1
)

# --- 2. LEARNING RATE SCHEDULER ---
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5,        
    patience=3,        
    min_lr=1e-6,       # Impedisce che il LR diventi microscopicamente inutile
    verbose=1
)

# --- 3. EARLY STOPPING ---
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=10,        
    restore_best_weights=True, # A fine addestramento, ricarica in automatico i pesi dell'epoca con la loss migliore!
    verbose=1
)

# FUOCO ALLE POLVERI
EPOCHS = 100 

print("\n--- INIZIO ADDESTRAMENTO OTTIMIZZATO (CON SCHEDULER E EARLY STOP) ---")
history_light = model_heavy.fit(
    train_gen,                
    validation_data=val_gen,  
    epochs=EPOCHS,
    callbacks=[checkpoint_heavy, reduce_lr, early_stop], # <--- Inseriti qui i nuovi array!
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")

In [ ]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V1-Heavy (Struttura Romana, Potenza Toscana) vecchia
# ==============================================================================
def build_eeai_model_v1_heavy(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # --- BLOCCO 1: Estrazione Base ---
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_1a")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x)
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_1b")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) # Dimensione: 120 -> 60
    
    # --- BLOCCO 2: Livello Intermedio ---
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_1")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_3")(x)
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_2")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_4")(x) # Dimensione: 60 -> 30
    
    # --- BLOCCO 3: Feature di Alto Livello ---
    x = layers.Conv2D(128, (1, 3), padding='same', activation='relu', name="conv_3a")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_5")(x)
    x = layers.Conv2D(128, (1, 3), padding='same', activation='relu', name="conv_3b")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_6")(x)
    x = layers.Conv2D(128, (1, 3), padding='same', activation='relu', name="conv_3c")(x)
    
    # Compattazione salvavita per ESP32
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    
    # --- COLLO DI BOTTIGLIA PIÙ PROFONDO ---
    x = layers.Dense(256, activation='relu', name="features_deep2")(x)
    x = layers.Dropout(0.2, name="drop_features")(x) 
    common_feat = layers.Dense(64, activation='relu', name="features")(x)

    # --- OUTPUT MULTI-HEAD INVARIATI ---
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    return models.Model(inputs=inputs, outputs=[coords_output, mask_output], name="EEAI_Net_V1_Heavy")

# Inizializzazione del nuovo modello
model_heavy = build_eeai_model_v1_heavy()

# Compilazione 
model_heavy.compile(
    optimizer='adam',
    loss={"coords_head": true_masked_mse, "mask_head": "binary_crossentropy"}, 
    loss_weights={"coords_head": 1.0, "mask_head": 0.5},
    metrics={
"coords_head": [true_masked_rmse_metres],
        "mask_head": [tf.keras.metrics.BinaryAccuracy(name="bin_acc")]
    }
)

# --- 1. IL CHECKPOINT ORIGINALE ---
checkpoint_heavy = ModelCheckpoint(
    "eeai_best_model_romano_heavy.keras", 
    monitor="val_loss", 
    save_best_only=True, 
    verbose=1
)

# --- 2. LEARNING RATE SCHEDULER ---
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5,        
    patience=3,        
    min_lr=1e-6,       # Impedisce che il LR diventi microscopicamente inutile
    verbose=1
)

# --- 3. EARLY STOPPING ---
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=10,        
    restore_best_weights=True, # A fine addestramento, ricarica in automatico i pesi dell'epoca con la loss migliore!
    verbose=1
)

# FUOCO ALLE POLVERI
EPOCHS = 100 

print("\n--- INIZIO ADDESTRAMENTO OTTIMIZZATO (CON SCHEDULER E EARLY STOP) ---")
history_light = model_heavy.fit(
    train_gen,                
    validation_data=val_gen,  
    epochs=EPOCHS,
    callbacks=[checkpoint_heavy, reduce_lr, early_stop], # <--- Inseriti qui i nuovi array!
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")

In [ ]:
# ==============================================================================
# MODEL SUMMARY PER EMBEDDED
# ==============================================================================

def embedded_summary(model, input_shape=(1, 120, 18)):
    
    # 2. Calcola i parametri statici (Flash)
    total_params = model.count_params()
    estimated_flash_kb = (total_params * 4) / 1024
    
   # 3. Calcola il picco di memoria dinamica (SRAM/Tensor Arena)
    max_layer_ram_kb = 0
    for layer in model.layers:
        # AGGIUNTO: Salta l'InputLayer o i layer senza output_shape per evitare l'AttributeError
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        layer_ram_kb = (num_elements * 4) / 1024
        if layer_ram_kb > max_layer_ram_kb:
            max_layer_ram_kb = layer_ram_kb

    input_elements = np.prod(input_shape)
    input_ram_kb = (input_elements * 4) / 1024
    peak_arena_kb = input_ram_kb + max_layer_ram_kb

    # 4. Stampa il verdetto 
    print("============================================")
    print("   REPORT REQUISITI HARDWARE (STIMA FLOAT32)   ")
    print("============================================")
    print(f" Memoria FLASH stimata : {estimated_flash_kb:.2f} KB  (Limite : < 800 KB)")
    print(f" Memoria SRAM stimata  : ~{peak_arena_kb:.2f} KB (Limite : < 400 KB)")
    print(" Operazioni Ricorrenti : ASSENTI (RNN/LSTM/GRU non rilevate)")
    print(" Nota sulla Quantizz.  : Raccomandata INT8 per ESP32-S3 (ridurrà la RAM di ~4x)")
    print("============================================\n")

embedded_summary(model_heavy)

In [ ]:
# ==============================================================================
# VISUALIZZATORE 3.0 (Anti-Sfarfallio e Ground Truth Fixata)
# ==============================================================================
#file_target = "dataset/data/window_000011.npz"
file_target = "dataset/window_000007.npz"

if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri e previsioni in corso (V2)...")
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.05
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)

    # Usa esplicitamente il modello V2 appena addestrato!
    preds = model_heavy.predict(decluttered, verbose=0)
    p_coords = preds[0].reshape(T, 4, 2)
    p_mask = preds[1]
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            ax.set_title(f"Radar V2 | Frame: {frame_idx}/{T-1} | Window: 07", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)